In [2]:
# table3_seedwise_selected.py
# 목적:
# - seed별 원본 결과(각 seed/run의 metric 값)를 저장
# - 모델/설정 제한:
#   * O  : MODEL = XGBoost
#   * F  : DAG = GES, MODEL = LightGBM
#   * OF : DAG = GES, MODEL = LightGBM
# - 각 Set에서 (MODEL, K_EDGE, N_FEAT, FEATURE_KEY [그리고 F/OF는 DAG]) 단위로
#   seed 평균 AUPRC가 최대인 1개 config를 선택
# - 그 config의 seed별 행(원본 metric)을 뽑아 ./detail에 저장
#
# 입력(./):
#   results_O_1.csv ... results_O_5.csv
#   results_F_1.csv ... results_F_5.csv
#   results_OF_1.csv ... results_OF_5.csv
#
# 출력(./detail):
#   table3_seedwise.csv              (seed별 원본)
#   table3_selected_configs.csv      (선택된 config 3개 요약)
#   table3_selected_meanstd.csv      (선택된 config의 mean/std 요약)

import os
import re
import glob
import pandas as pd

METRICS = ["AUROC", "AUPRC", "F1", "Brier", "ECE"]
BASE_REQUIRED = ["SET", "DAG", "MODEL", "K_EDGE", "N_FEAT", "FEATURE_KEY"] + METRICS


def load_results(pattern: str) -> pd.DataFrame:
    paths = sorted(glob.glob(pattern))
    if not paths:
        raise FileNotFoundError(f"No files matched: {pattern}")

    dfs = []
    for p in paths:
        df = pd.read_csv(p)

        m = re.search(r"_(\d+)\.csv$", os.path.basename(p))
        df["SEED_RUN"] = int(m.group(1)) if m else None
        df["SOURCE_FILE"] = os.path.basename(p)

        dfs.append(df)

    out = pd.concat(dfs, ignore_index=True)

    missing = [c for c in BASE_REQUIRED if c not in out.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    # numeric cast (robust)
    out["K_EDGE"] = pd.to_numeric(out["K_EDGE"], errors="coerce")
    out["N_FEAT"] = pd.to_numeric(out["N_FEAT"], errors="coerce")
    for m in METRICS:
        out[m] = pd.to_numeric(out[m], errors="coerce")

    # drop rows missing essentials
    out = out.dropna(subset=["K_EDGE", "N_FEAT"] + METRICS).copy()
    out["K_EDGE"] = out["K_EDGE"].astype(int)
    out["N_FEAT"] = out["N_FEAT"].astype(int)

    return out


def select_best_config(df: pd.DataFrame, config_cols: list) -> pd.Series:
    """
    df: already filtered to desired (SET/DAG/MODEL subset)
    config_cols: columns that define a config
    Select config with max AUPRC_mean across seeds.
    """
    if df.empty:
        raise ValueError("Empty df after filtering. Check filters / input files.")

    agg = (
        df.groupby(config_cols)[METRICS]
          .agg(["mean", "std", "count"])
          .reset_index()
    )

    # flatten columns
    agg.columns = [
        "_".join([x for x in col if x]) if isinstance(col, tuple) else col
        for col in agg.columns
    ]

    # sort by AUPRC_mean desc
    best = agg.sort_values("AUPRC_mean", ascending=False).head(1)
    if best.empty:
        raise ValueError("Could not select best config (no rows).")

    return best.iloc[0]


def extract_seedwise_rows(df: pd.DataFrame, best_cfg: dict, match_cols: list) -> pd.DataFrame:
    """
    df: filtered df that includes all seeds
    best_cfg: dict with keys in match_cols
    match_cols: list of columns to match exactly
    Returns seedwise rows. If multiple rows per seed, pick max AUPRC row.
    """
    q = pd.Series([True] * len(df), index=df.index)
    for c in match_cols:
        q &= (df[c] == best_cfg[c])
    sub = df[q].copy()

    if sub.empty:
        raise ValueError(f"No seedwise rows matched best config: {best_cfg}")

    # If duplicates per seed, keep the best AUPRC row per seed
    sub = (
        sub.sort_values(["SEED_RUN", "AUPRC"], ascending=[True, False])
           .groupby("SEED_RUN", as_index=False)
           .head(1)
    )
    return sub


def meanstd_of_seedwise(seedwise: pd.DataFrame) -> dict:
    out = {}
    for m in METRICS:
        out[f"{m}_mean"] = float(seedwise[m].mean())
        out[f"{m}_std"] = float(seedwise[m].std(ddof=1)) if len(seedwise) > 1 else 0.0
        out[f"{m}_count"] = int(seedwise[m].count())
    return out


def main():
    OUT_DIR = "./detail"
    os.makedirs(OUT_DIR, exist_ok=True)

    # Load
    df_O  = load_results("./results_O_*.csv")
    df_F  = load_results("./results_F_*.csv")
    df_OF = load_results("./results_OF_*.csv")

    df_all = pd.concat([df_O, df_F, df_OF], ignore_index=True)

    # =========================
    # 1) Filter by your constraints
    # =========================
    # O: XGBoost only
    df_O_sel = df_all[(df_all["SET"] == "O") & (df_all["MODEL"] == "XGBoost")].copy()

    # F: GES + LightGBM only
    df_F_sel = df_all[(df_all["SET"] == "F") & (df_all["DAG"] == "GES") & (df_all["MODEL"] == "LightGBM")].copy()

    # OF: GES + LightGBM only
    df_OF_sel = df_all[(df_all["SET"] == "OF") & (df_all["DAG"] == "GES") & (df_all["MODEL"] == "LightGBM")].copy()

    # =========================
    # 2) Select best config within each subset by AUPRC_mean
    # =========================
    # Config definition columns
    cfg_cols_O  = ["MODEL", "K_EDGE", "N_FEAT", "FEATURE_KEY"]
    cfg_cols_F  = ["DAG", "MODEL", "K_EDGE", "N_FEAT", "FEATURE_KEY"]
    cfg_cols_OF = ["DAG", "MODEL", "K_EDGE", "N_FEAT", "FEATURE_KEY"]

    best_O  = select_best_config(df_O_sel,  cfg_cols_O)
    best_F  = select_best_config(df_F_sel,  cfg_cols_F)
    best_OF = select_best_config(df_OF_sel, cfg_cols_OF)

    best_cfg_O = {
        "MODEL": best_O["MODEL"],
        "K_EDGE": int(best_O["K_EDGE"]),
        "N_FEAT": int(best_O["N_FEAT"]),
        "FEATURE_KEY": best_O["FEATURE_KEY"],
    }
    best_cfg_F = {
        "DAG": best_F["DAG"],
        "MODEL": best_F["MODEL"],
        "K_EDGE": int(best_F["K_EDGE"]),
        "N_FEAT": int(best_F["N_FEAT"]),
        "FEATURE_KEY": best_F["FEATURE_KEY"],
    }
    best_cfg_OF = {
        "DAG": best_OF["DAG"],
        "MODEL": best_OF["MODEL"],
        "K_EDGE": int(best_OF["K_EDGE"]),
        "N_FEAT": int(best_OF["N_FEAT"]),
        "FEATURE_KEY": best_OF["FEATURE_KEY"],
    }

    # =========================
    # 3) Extract seedwise rows for those configs
    # =========================
    seed_O  = extract_seedwise_rows(df_O_sel,  best_cfg_O,  ["MODEL", "K_EDGE", "N_FEAT", "FEATURE_KEY"])
    seed_F  = extract_seedwise_rows(df_F_sel,  best_cfg_F,  ["DAG", "MODEL", "K_EDGE", "N_FEAT", "FEATURE_KEY"])
    seed_OF = extract_seedwise_rows(df_OF_sel, best_cfg_OF, ["DAG", "MODEL", "K_EDGE", "N_FEAT", "FEATURE_KEY"])

    # Add display fields
    def decorate(seed_df: pd.DataFrame, feature_set: str, dag_method: str, best_model: str) -> pd.DataFrame:
        x = seed_df.copy()
        x["Feature Set"] = feature_set
        x["DAG Method"] = dag_method
        x["Best Model"] = best_model
        return x

    seed_O  = decorate(seed_O,  "O",  "–",   "XGBoost")
    seed_F  = decorate(seed_F,  "F",  "GES", "LightGBM")
    seed_OF = decorate(seed_OF, "OF", "GES", "LightGBM")

    # Keep only useful columns
    cols_out = [
        "SEED_RUN", "Feature Set", "DAG Method", "Best Model",
        "K_EDGE", "N_FEAT", "FEATURE_KEY",
        "AUROC", "AUPRC", "F1", "Brier", "ECE",
        "SOURCE_FILE"
    ]

    table_seedwise = pd.concat(
        [seed_O[cols_out], seed_F[cols_out], seed_OF[cols_out]],
        ignore_index=True
    ).sort_values(["Feature Set", "DAG Method", "SEED_RUN"], ascending=[True, True, True])

    # =========================
    # 4) Save seedwise
    # =========================
    out_seedwise = os.path.join(OUT_DIR, "table3_seedwise.csv")
    table_seedwise.to_csv(out_seedwise, index=False, encoding="utf-8-sig")

    # =========================
    # 5) Save selected configs + mean/std summary (optional but helpful)
    # =========================
    configs = []
    configs.append({"Feature Set": "O", "DAG Method": "–",   **best_cfg_O})
    configs.append({"Feature Set": "F", "DAG Method": "GES", **best_cfg_F})
    configs.append({"Feature Set": "OF","DAG Method": "GES", **best_cfg_OF})
    df_configs = pd.DataFrame(configs)

    out_configs = os.path.join(OUT_DIR, "table3_selected_configs.csv")
    df_configs.to_csv(out_configs, index=False, encoding="utf-8-sig")

    # mean/std computed from extracted seedwise rows
    sum_rows = []
    for feature_set, dag_method, seed_df, cfg in [
        ("O",  "–",   seed_O,  best_cfg_O),
        ("F",  "GES", seed_F,  best_cfg_F),
        ("OF", "GES", seed_OF, best_cfg_OF),
    ]:
        s = {"Feature Set": feature_set, "DAG Method": dag_method, **cfg}
        s.update(meanstd_of_seedwise(seed_df))
        sum_rows.append(s)

    df_meanstd = pd.DataFrame(sum_rows)
    out_meanstd = os.path.join(OUT_DIR, "table3_selected_meanstd.csv")
    df_meanstd.to_csv(out_meanstd, index=False, encoding="utf-8-sig")

    # Print
    print(f"[OK] Saved seedwise: {out_seedwise}")
    print(f"[OK] Saved configs : {out_configs}")
    print(f"[OK] Saved mean/std : {out_meanstd}")
    print()
    print("=== Selected configs ===")
    print(df_configs.to_string(index=False))
    print()
    print("=== Seedwise preview (top 30 rows) ===")
    print(table_seedwise.head(30).to_string(index=False))


if __name__ == "__main__":
    main()

[OK] Saved seedwise: ./detail\table3_seedwise.csv
[OK] Saved configs : ./detail\table3_selected_configs.csv
[OK] Saved mean/std : ./detail\table3_selected_meanstd.csv

=== Selected configs ===
Feature Set DAG Method    MODEL  K_EDGE  N_FEAT      FEATURE_KEY DAG
          O          –  XGBoost       0      13 d142655db65fda0d NaN
          F        GES LightGBM      34      34 b154670abb75bb96 GES
         OF        GES LightGBM      34      47 084da68000a6f7f1 GES

=== Seedwise preview (top 30 rows) ===
 SEED_RUN Feature Set DAG Method Best Model  K_EDGE  N_FEAT      FEATURE_KEY    AUROC    AUPRC       F1    Brier      ECE      SOURCE_FILE
        1           F        GES   LightGBM      34      34 b154670abb75bb96 0.928259 0.472686 0.452055 0.016727 0.016313  results_F_1.csv
        2           F        GES   LightGBM      34      34 b154670abb75bb96 0.924445 0.471468 0.458599 0.016684 0.016592  results_F_2.csv
        3           F        GES   LightGBM      34      34 b154670abb75bb